In [1]:
!pip install --no-index /kaggle/input/datasets/kurshidbasheer/biopython-offline/biopython-1.83-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

Processing /kaggle/input/datasets/kurshidbasheer/biopython-offline/biopython-1.83-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl


In [2]:
!pip install --no-index /kaggle/input/datasets/kurshidbasheer/pyg-2-7-torch-2-9-cpu-py312-kur/torch_geometric-2.7.0-py3-none-any.whl

Processing /kaggle/input/datasets/kurshidbasheer/pyg-2-7-torch-2-9-cpu-py312-kur/torch_geometric-2.7.0-py3-none-any.whl


In [3]:
# =========================
# IMPORTS
# =========================
import torch, random
import numpy as np
import pandas as pd
import torch.nn as nn
from collections import defaultdict

from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.utils import scatter

In [4]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TRAIN_SEQ = "/kaggle/input/competitions/stanford-rna-3d-folding-2/train_sequences.csv"
TRAIN_LBL = "/kaggle/input/competitions/stanford-rna-3d-folding-2/train_labels.csv"
TEST_SEQ  = "/kaggle/input/competitions/stanford-rna-3d-folding-2/test_sequences.csv"

WINDOW_SIZE = 200
STRIDE = 100
K_NEIGHBORS = 12   # 🔥 reduced → faster

In [5]:
# =========================
# UTILS
# =========================
NUC_MAP = {'A':0, 'U':1, 'G':2, 'C':3}

def clean_sequence(seq):
    return "".join([s for s in seq.upper() if s in NUC_MAP])

def one_hot(seq):
    x = torch.zeros(len(seq), 4)
    for i, s in enumerate(seq):
        x[i, NUC_MAP[s]] = 1
    return x

In [6]:
# =========================
# GRAPH (FAST)
# =========================
def build_graph(x, coords=None, k=12):
    L = x.size(0)

    row = []
    col = []

    for i in range(L):
        neigh = list(range(max(0,i-k), min(L,i+k+1)))
        if i in neigh: neigh.remove(i)
        if len(neigh) > k:
            neigh = random.sample(neigh, k)

        row += [i]*len(neigh)
        col += neigh

    edge_index = torch.tensor([row,col], dtype=torch.long)

    rel = (edge_index[0] - edge_index[1]).float().unsqueeze(1) / L
    dist = rel.abs()

    edge_attr = torch.cat([dist, rel], dim=1)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

    if coords is not None:
        data.pos = coords
        data.y   = coords
    else:
        pos = torch.arange(L).float().unsqueeze(1).repeat(1,3)
        data.pos = pos * 0.01

    return data


In [7]:
# =========================
# DATASET (PREBUILT GRAPH)
# =========================
class RNAWindowDataset(Dataset):
    def __init__(self, seq_csv, label_csv=None):
        self.df = pd.read_csv(seq_csv)
        self.has_labels = label_csv is not None

        # ✅ ADDED: sequence cache
        self.seq_map = {
            row["target_id"]: clean_sequence(row["sequence"])
            for _, row in self.df.iterrows()
        }

        if self.has_labels:
            labels = pd.read_csv(label_csv, low_memory=False)
            labels["sid"] = labels["ID"].str.split("_").str[0]
            labels["idx"] = labels["ID"].str.split("_").str[1].astype(int)

            self.coords = {}
            for k,g in labels.groupby("sid"):
                g = g.sort_values("idx")
                xyz = torch.tensor(g[["x_1","y_1","z_1"]].values, dtype=torch.float32)

                valid = ~torch.isnan(xyz).any(dim=1)
                xyz = xyz[valid]

                if len(xyz)>0:
                    xyz = xyz - xyz.mean(0,keepdim=True)
                    xyz = xyz/(xyz.std()+1e-8)
                    self.coords[k] = xyz

        self.samples = []

        for sid in self.df["target_id"]:
            # ✅ CHANGED: use cached sequence
            seq = self.seq_map[sid]

            L = len(seq)
            if self.has_labels and sid in self.coords:
                L = min(L, self.coords[sid].shape[0])

            for s in range(0, L, STRIDE):
                e = min(s+WINDOW_SIZE, L)
                if e-s >= 10:
                    self.samples.append((sid,s,e))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sid,s,e = self.samples[idx]

        # ✅ CHANGED: use cached sequence
        seq = self.seq_map[sid]

        coords = None
        if self.has_labels and sid in self.coords:
            coords = self.coords[sid][:len(seq)][s:e]

        seq = seq[s:e]

        x = one_hot(seq)
        pos_feat = torch.arange(len(seq)).float().unsqueeze(-1)/len(seq)
        x = torch.cat([x,pos_feat],dim=1)

        return build_graph(x, coords)

In [8]:
# =========================
# KABSCH LOSS
# =========================
def kabsch(P, Q):
    Pc = P - P.mean(0, keepdim=True)
    Qc = Q - Q.mean(0, keepdim=True)

    C = Pc.t() @ Qc

    U, S, Vt = torch.linalg.svd(C)

    d = torch.det(U @ Vt)
    D = torch.eye(3, device=P.device)
    D[-1, -1] = d

    R = U @ D @ Vt

    return Pc @ R
    

def geometric_loss(pred, target, batch):
    loss = 0
    n = batch.max()+1

    for i in range(n):
        mask = batch == i

        P = pred[mask].float()
        Q = target[mask].float()

        # 🔥 CRITICAL FIX
        with torch.amp.autocast("cuda", enabled=False):
            P = kabsch(P, Q)

        coord = ((P - Q)**2).mean()

        d1 = torch.cdist(P, P)
        d2 = torch.cdist(Q, Q)

        dist = ((d1 - d2)**2).mean()

        loss += coord + 0.1 * dist

    return loss/n

In [9]:
# =========================
# EGNN
# =========================
class EGNNLayer(nn.Module):
    def __init__(self, hidden):
        super().__init__()

        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden*2+2, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden)
        )

        self.node_mlp = nn.Sequential(
            nn.Linear(hidden*2, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden)
        )

        self.coord_mlp = nn.Sequential(
            nn.Linear(hidden,1),
            nn.Tanh()
        )

    def forward(self,x,pos,edge_index,edge_attr):
        row,col = edge_index

        rel = pos[row]-pos[col]

        m = self.edge_mlp(torch.cat([x[row],x[col],edge_attr],dim=1))

        agg = scatter(m,row,dim=0,dim_size=x.size(0),reduce="mean")
        x = self.node_mlp(torch.cat([x,agg],dim=1))

        trans = self.coord_mlp(m)*rel
        delta = scatter(trans,row,dim=0,dim_size=pos.size(0),reduce="mean")

        pos = pos + delta
        return x,pos

class EGNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Linear(5,64)
        self.layers = nn.ModuleList([EGNNLayer(64) for _ in range(4)])

    def forward(self,data):
        x,pos = data.x,data.pos
        x = self.emb(x)

        for l in self.layers:
            x,pos = l(x,pos,data.edge_index,data.edge_attr)

        return pos

In [10]:
# =========================
# TRAIN (AMP)
# =========================
def train_epoch(model,loader,opt,scaler):
    model.train()
    total=0

    for data in loader:
        data = data.to(DEVICE)

        opt.zero_grad()

        with torch.cuda.amp.autocast():
            pred = model(data)
            loss = geometric_loss(pred,data.y,data.batch)

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        total += loss.item()

    return total/len(loader)

In [11]:
# =========================
# INFERENCE
# =========================
def run_inference(model, dataset):
    model.eval()
    storage = defaultdict(list)

    for sid,s,e in dataset.samples:
        row = dataset.df[dataset.df["target_id"]==sid].iloc[0]
        seq = clean_sequence(row["sequence"])[s:e]

        x = one_hot(seq)
        pos_feat = torch.arange(len(seq)).float().unsqueeze(-1)/len(seq)
        x = torch.cat([x,pos_feat],dim=1)

        g = build_graph(x, None)
        data = g.to(DEVICE)

        with torch.no_grad():
            pred = model(data).cpu()

        storage[sid].append((s,e,pred))

    return storage

def merge_windows(storage):
    final={}

    for sid,chunks in storage.items():
        L = max(e for _,e,_ in chunks)

        coords = torch.zeros(L,3)
        counts = torch.zeros(L,1)

        for s,e,p in chunks:
            w = torch.linspace(0.5,1.0,e-s).unsqueeze(1)
            coords[s:e]+=p*w
            counts[s:e]+=w

        final[sid]=coords/counts.clamp(min=1)

    return final

In [12]:
# =========================
# SUBMISSION
# =========================
def build_submission(test_ds, preds):
    rows=[]

    for sid in test_ds.df["target_id"]:
        row = test_ds.df[test_ds.df["target_id"]==sid].iloc[0]
        seq = clean_sequence(row["sequence"])

        coords = preds[sid]

        for i in range(len(seq)):
            r={"ID":f"{sid}_{i+1}","resname":seq[i],"resid":i+1}

            for k in range(5):
                noise = 0.02*torch.randn_like(coords)
                c = coords+noise

                r[f"x_{k+1}"]=float(c[i,0])
                r[f"y_{k+1}"]=float(c[i,1])
                r[f"z_{k+1}"]=float(c[i,2])

            rows.append(r)

    pd.DataFrame(rows).to_csv("submission.csv",index=False)

In [14]:
from tqdm.auto import tqdm
tqdm.monitor_interval = 0  # 🔥 fixes Jupyter/Kaggle spam

import torch
from torch_geometric.loader import DataLoader  # ✅ IMPORTANT (PyG loader)

# =========================
# TRAIN FUNCTION
# =========================
def train_epoch(model, loader, opt, scaler, epoch=0):
    model.train()
    total = 0

    pbar = tqdm(
        loader,
        desc=f"Epoch {epoch}",
        leave=True,
        dynamic_ncols=True,
        mininterval=0.5
    )

    for i, data in enumerate(pbar):
        data = data.to(DEVICE)

        opt.zero_grad()

        with torch.amp.autocast("cuda"):
            pred = model(data)
            loss = geometric_loss(pred, data.y, data.batch)

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        total += loss.item()

        # ✅ update occasionally (no spam)
        if i % 100 == 0:
            pbar.set_postfix({"loss": f"{loss.item():.4f}"}, refresh=False)

    return total / len(loader)


# =========================
# DATA
# =========================
train_ds = RNAWindowDataset(TRAIN_SEQ, TRAIN_LBL)
test_ds  = RNAWindowDataset(TEST_SEQ, None)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)


# =========================
# MODEL + OPTIM
# =========================
model = EGNNModel().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
scaler = torch.amp.GradScaler("cuda")


# =========================
# TRAIN LOOP
# =========================
EPOCHS = 5

for e in range(EPOCHS):
    loss = train_epoch(model, train_loader, opt, scaler, epoch=e)
    print(f"Epoch {e} done | avg loss = {loss:.8f}")


# =========================
# INFERENCE
# =========================
print("Inference...")
model.eval()

storage = run_inference(model, test_ds)

print("Merging...")
merged = merge_windows(storage)

print("Submission...")
build_submission(test_ds, merged)

print("DONE ✅")

Epoch 0:   0%|          | 0/9480 [00:00<?, ?it/s]

Epoch 0 done | avg loss = 0.62378714


Epoch 1:   0%|          | 0/9480 [00:00<?, ?it/s]

Epoch 1 done | avg loss = 0.62377554


Epoch 2:   0%|          | 0/9480 [00:00<?, ?it/s]

Epoch 2 done | avg loss = 0.62376958


Epoch 3:   0%|          | 0/9480 [00:00<?, ?it/s]

Epoch 3 done | avg loss = 0.62379830


Epoch 4:   0%|          | 0/9480 [00:00<?, ?it/s]

Epoch 4 done | avg loss = 0.62381274
Inference...
Merging...
Submission...
DONE ✅
